# EPIC Clarity Death Hydration

This notebook hydrates the OMOP DEATH table from EPIC Clarity patient death dates.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT` - Primary death_date field
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_4` - External/supplemental death_date source

## OMOP Fields Populated
- person_id
- death_date (coalesce internal and external sources)
- death_type_concept_id

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC death data%sqlCREATE OR REPLACE TEMP VIEW death_silver ASSELECT    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS person_source_value,    COALESCE(p.DEATH_DATE, p4.EXTERNAL_DEATH_DATE) AS death_date,    CURRENT_TIMESTAMP() AS updated_tspFROM _exponent._bronze_epic_clarity.patient pLEFT JOIN _exponent._bronze_epic_clarity.patient_3 p4    ON p.PAT_ID = p4.PAT_IDWHERE p.DEATH_DATE IS NOT NULL OR p4.EXTERNAL_DEATH_DATE IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.death AS target
USING death_silver AS source
ON target.person_source_value = source.person_source_value

WHEN MATCHED AND NOT (
    target.death_date <=> source.death_date
)
THEN UPDATE SET
    target.death_date = source.death_date,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    person_source_value,
    death_date,
    updated_tsp
)
VALUES (
    source.person_source_value,
    source.death_date,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_death (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.person_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT person_source_value, updated_tsp
    FROM _exponent.omop_silver.death
    WHERE person_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_death x
    ON s.person_source_value = x.person_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with person mapping to get person_id
%sql
CREATE OR REPLACE TEMP VIEW death_gold AS
SELECT
    m.person_id,
    s.death_date,
    s.updated_tsp
FROM _exponent.omop_silver.death s
INNER JOIN _exponent.omop_mapping.source_to_person m
    ON s.person_source_value = m.person_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.death AS target
USING death_gold AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
    target.death_date <=> source.death_date
)
THEN UPDATE SET
    target.death_date = source.death_date

WHEN NOT MATCHED THEN INSERT (
    person_id,
    death_date,
    death_type_concept_id
)
VALUES (
    source.person_id,
    source.death_date,
    0  -- Unknown death type
)